# Intro to PandaPower for IDP Project Optimization

In [1]:
try:
    import matplotlib as plt
    from pandapower.networks.power_system_test_cases import case30
    from pandapower.run import runpp, runopp
    from pandapower.timeseries.run_time_series import run_timeseries, OutputWriter
    from pandapower.diagnostic import Diagnostic
    from pandapower.create import create_storage

except: 
    %pip install pandapower
    %pip install numba
    %pip install matplotlib
    import matplotlib as plt
    from pandapower.networks.power_system_test_cases import case30
    from pandapower.run import runpp, runopp
    from pandapower.timeseries.run_time_series import run_timeseries

from tools.limits import bus_vm_pu_limits, line_loading_limits
import profileloader as pl
import numpy as np
import os
import tempfile

### Import IEEE Case 30:
Case 30 is stored in a variable, "net", which consists of several dataframes. 
You can conceptualize a dataframe as being like a spreadsheet: It has named rows and columns, and data is stored in individual cells for each row/column pair. 
By default, the IEEE 30 net has a dataframe for each of the following grid components
- bus (30)
- load (21)
- generator (5)
- shunt (2)
- ext_grid (1)
- line (34)
- transformer (7)
- cost (6)

Additional components, such as for storage, motors, etc. can also be added to the net

In [2]:
net = case30()
print(net)

This pandapower network includes the following parameter tables:
   - bus (30 elements)
   - load (20 elements)
   - gen (5 elements)
   - shunt (2 elements)
   - ext_grid (1 element)
   - line (41 elements)
   - poly_cost (6 elements)


In [3]:
"""store_el = create_storage(
    net, 3, 
    p_mw = .1, 
    q_mvar = 0, 
    max_e_mwh = 0.1, 
    controllable=True, 
    max_p_mw=0.1, 
    min_p_mw=-0.1, 
    max_q_mvar=0.1,
    min_q_mvar=-0.1)
print(net)"""

'store_el = create_storage(\n    net, 3, \n    p_mw = .1, \n    q_mvar = 0, \n    max_e_mwh = 0.1, \n    controllable=True, \n    max_p_mw=0.1, \n    min_p_mw=-0.1, \n    max_q_mvar=0.1,\n    min_q_mvar=-0.1)\nprint(net)'

In [4]:
net.storage

,name,bus,p_mw,q_mvar,sn_mva,soc_percent,min_e_mwh,max_e_mwh,scaling,in_service,type


Specific cells are adjustable based on external data

In [5]:
net.line.head()

,name,std_type,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,g_us_per_km,max_i_ka,df,parallel,type,in_service,max_loading_percent,geo
0,None,None,0,1,1.0,3.6450,10.9350,436.639076,0.0,0.555967,1.0,1,ol,True,100.0,None
1,None,None,0,2,1.0,9.1125,34.6275,291.092717,0.0,0.555967,1.0,1,ol,True,100.0,None
2,None,None,1,3,1.0,10.9350,30.9825,291.092717,0.0,0.277983,1.0,1,ol,True,100.0,None
3,None,None,2,3,1.0,1.8225,7.2900,0.000000,0.0,0.555967,1.0,1,ol,True,100.0,None
4,None,None,1,4,1.0,9.1125,36.4500,291.092717,0.0,0.555967,1.0,1,ol,True,100.0,None


In [6]:
# for example: set the power consumption of load #1 to 2x its starting value
print("Before: " + str(net.load.at[0,'p_mw']))
net.load.at[0,'p_mw'] = 2 * net.load.iloc[0].p_mw
print("After: " + str(net.load.at[0,'p_mw']))

# and: set line loading limit to 50%
print("Before: " + str(net.line.at[10,'max_loading_percent']))
net.line.at[10,'max_loading_percent'] = 0.5
print("After: " + str(net.line.at[10,'max_loading_percent']))

# and: set generator 5 to limit to 2x its starting value
print("Before: " + str(net.gen.at[4,'p_mw']))
net.gen.at[4,'p_mw'] = 5 * net.gen.iloc[4].p_mw
print("After: " + str(net.gen.at[4,'p_mw']))

Before: 21.7
After: 43.4
Before: 100.0
After: 0.5
Before: 37.0
After: 185.0


### Power flow analysis
Analysis can be run on net using the runpp function. This adds a new set of dataframes to the net, containing result data for each bus, load, generator, etc.

In [7]:
runpp(net)
net.res_line.head()

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,-58.987589,19.468940,59.771435,-20.117401,0.783846,-0.648461,0.265656,0.269713,0.269713,1.000000,0.000000,1.000000,2.268723,48.512392
1,-33.522925,15.040468,34.213466,-14.394311,0.690542,0.646158,0.157135,0.160526,0.160526,1.000000,0.000000,0.988888,4.158718,28.873257
2,-22.920998,15.216666,23.394010,-15.851398,0.473011,-0.634732,0.117660,0.122396,0.122396,1.000000,2.268723,0.987386,5.095572,44.030121
3,-36.613466,13.194311,36.768353,-12.574764,0.154887,0.619546,0.168311,0.168311,0.168311,0.988888,4.158718,0.987386,5.095572,30.273560
4,-2.755707,7.873832,2.798876,-9.668782,0.043169,-1.794950,0.035677,0.043762,0.043762,1.000000,2.268723,0.983680,2.848187,7.871330


You can use the result dataframes to check if power flow data is within the defined limits for each object

In [8]:
# For example, buses:
bus_vm_pu_limits(net)
line_loading_limits(net)

All buses within limits
Line #9: Result 107.7809% loading is greater than maximum of 100.0% loading
Line #10: Result 34.1365% loading is greater than maximum of 0.5% loading
Line #14: Result 144.8378% loading is greater than maximum of 100.0% loading
Line #15: Result 291.7905% loading is greater than maximum of 100.0% loading
Line #17: Result 117.3544% loading is greater than maximum of 100.0% loading
Line #18: Result 119.3227% loading is greater than maximum of 100.0% loading
Line #20: Result 221.111% loading is greater than maximum of 100.0% loading
Line #21: Result 151.2633% loading is greater than maximum of 100.0% loading
Line #22: Result 132.7413% loading is greater than maximum of 100.0% loading
Line #28: Result 124.1101% loading is greater than maximum of 100.0% loading
Line #29: Result 122.7833% loading is greater than maximum of 100.0% loading
Line #31: Result 146.02% loading is greater than maximum of 100.0% loading


For purposes of optimization, grid components such as generators and loads have a boolean property called "controllable". If a component's "controllable" property is true, that means that the component can be changed during the optimization algorithm, to find which values produce the optimal grid conditions. Values where "controllable" is false are those which must be kept fixed.  
By default, all sources are controllable, and all loads are not. 

In [9]:
try: 
    runopp(net)
except: 
    diag = Diagnostic()
    result = diag.diagnose_network(net, report_style="detailed")    


Below, you can see that the controllable grids have had their p_mw values changed. Compare the generator results with the generator results from before:

In [10]:
net.res_gen.head()

,p_mw,q_mvar,va_degree,vm_pu
0,53.623339,25.937628,-0.964218,1.001399
1,28.286298,7.339579,-1.613982,0.994606
2,44.732117,29.905295,1.609648,1.068951
3,24.099484,9.353275,-0.788258,1.034163
4,22.954928,44.698565,-0.772041,1.086644


In [11]:
# proof that bus max/min values are now accounted for
bus_vm_pu_limits(net)
line_loading_limits(net)

All buses within limits
All line loading within limits


To demonstrate optimization working if we set one of the generators to a fixed, non-controllable value: 

In [12]:
net.gen.at[0,'p_mw'] = 20
net.gen.at[0, 'controllable'] = False
net.gen

,name,bus,p_mw,vm_pu,sn_mva,min_q_mvar,max_q_mvar,scaling,slack,in_service,slack_weight,type,controllable,max_p_mw,min_p_mw,id_q_capability_characteristic,reactive_capability_curve,curve_style
0,None,1,20.00,1.0,NaN,-20.0,60.0,1.0,False,True,0.0,None,False,80.0,0.0,<NA>,False,None
1,None,21,21.59,1.0,NaN,-15.0,62.5,1.0,False,True,0.0,None,True,50.0,0.0,<NA>,False,None
2,None,26,26.91,1.0,NaN,-15.0,48.7,1.0,False,True,0.0,None,True,55.0,0.0,<NA>,False,None
3,None,22,19.20,1.0,NaN,-10.0,40.0,1.0,False,True,0.0,None,True,30.0,0.0,<NA>,False,None
4,None,12,185.00,1.0,NaN,-15.0,44.7,1.0,False,True,0.0,None,True,40.0,0.0,<NA>,False,None


In [13]:
runpp(net)
bus_vm_pu_limits(net)
line_loading_limits(net)

All buses within limits
Line #9: Result 107.7641% loading is greater than maximum of 100.0% loading
Line #10: Result 34.2844% loading is greater than maximum of 0.5% loading
Line #14: Result 144.4815% loading is greater than maximum of 100.0% loading
Line #15: Result 291.7512% loading is greater than maximum of 100.0% loading
Line #17: Result 117.576% loading is greater than maximum of 100.0% loading
Line #18: Result 119.5954% loading is greater than maximum of 100.0% loading
Line #20: Result 221.6465% loading is greater than maximum of 100.0% loading
Line #21: Result 151.5693% loading is greater than maximum of 100.0% loading
Line #22: Result 133.0472% loading is greater than maximum of 100.0% loading
Line #28: Result 124.0475% loading is greater than maximum of 100.0% loading
Line #29: Result 122.821% loading is greater than maximum of 100.0% loading
Line #31: Result 146.3772% loading is greater than maximum of 100.0% loading


In [14]:
runopp(net)
bus_vm_pu_limits(net)
line_loading_limits(net)


All buses within limits
All line loading within limits


The resulting optimized components have changed based on this new constraint. 

In [15]:
net.res_gen


,p_mw,q_mvar,va_degree,vm_pu
0,20.000000,41.437438,-1.671171,1.000000
1,26.676656,22.505415,-1.559991,0.993481
2,54.393313,19.707795,3.805768,1.038758
3,24.579061,7.077400,0.134865,1.014102
4,28.788108,27.608108,0.685700,1.040266


### Analysis with Custom Data

Load in custom loads/gens from the provided Excel sheets

In [16]:
load_control = pl.json_to_net(net, "load", limit=1000)

In [17]:
ow = OutputWriter(net, 
                  output_path="C:\\Users\\victor\\Documents\\College\\Fachhochschule Oberosterreich\\Interdisciplinary_Project\\idp-grid-stability", 
                  output_file_type='.csv', csv_separator=",")
run_timeseries(net, continue_on_divergence=True)

No time steps to calculate are specified. I'll check the datasource of the first controller for avaiable time steps
100%|█████████▉| 996/1000 [00:12<00:00, 81.89it/s]

In [18]:
output_dir = os.path.join(tempfile.gettempdir(), "gridtest")
print(output_dir)

C:\Users\victor\AppData\Local\Temp\gridtest


100%|██████████| 1000/1000 [00:30<00:00, 81.89it/s]